# Transferring cell type and niche annotations from H&E

Cell type and niche annotations usually come from spatial transcriptomics. SIGMMA's
image encoder was aligned with ST during training, so its H&E embeddings carry enough
molecular context to predict these annotations **from H&E only**. Annotations from a
profiled region can then be transferred to tissue that only has H&E.

We use the public 10x Xenium pancreatic adenocarcinoma section `TENX116` (HEST-1k), which
comes with 10x's cell type annotation (`cell_groups.csv`). In this tutorial we:

1. define spatial niches from the neighbourhood composition of the annotated cell types,
2. crop each cell from the H&E at three field sizes (micro / meso / macro),
3. embed the crops with SIGMMA's image encoder and, as a baseline, with raw UNI,
4. train a linear classifier on the left part of the section and predict cell types and
   niches on the right part, which is never seen during training.

**Requirements:** a GPU, the UNI2-h weights, a SIGMMA checkpoint (see {doc}`../installation`),
and for the section the H&E zarr, `spatial_data_processed.h5`, `cell_boundaries.parquet`
and `cell_groups.csv` from the Xenium output.

## Paths

In [ ]:
from pathlib import Path

SECTION = "TENX116"
SECTION_DIR = Path("/lustre/scratch126/cellgen/lotfollahi/projects/sigmma/data/public_xenium/hest_1k_pancreas") / SECTION
CELL_GROUPS = Path("/lustre/scratch126/cellgen/lotfollahi/dj16/data/public_xenium/hest_1k_PAAD/TENX116/cell_groups.csv")
CKPT = Path("/lustre/scratch126/cellgen/lotfollahi/dj16/projects/sigmma_outputs/checkpoints/"
            "sigmma_dense_attnpool_gex_svg_a01/36852/checkpoint-step2000.pth")
UNI_DIR = Path("/nfs/team361/dj16/projects/SIGMMA/assets/ckpts/uni_h/")

OUT = Path("sigmma_tutorial_outputs/cell_niche")
for sub in ("data", "sigmma", "uni"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import anndata as ad
import h5py
import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import torch
from ml_collections.config_dict import ConfigDict
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

from sigmma.utils import load_model
from sigmma.model.models_image_backbones import UNI
from sigmma.application.cell_niche_transfer import process_slide_data, process_slide_features

plt.rcParams["figure.dpi"] = 80
device = torch.device("cuda")

## 1. Annotations: cell types and niches

`cell_groups.csv` assigns each cell to one of 16 cell types. We define niches as groups of
cells with a similar local cell type composition: for every cell, count the cell types
within 50 µm and cluster these composition vectors with k-means.

In [ ]:
with h5py.File(SECTION_DIR / "spatial_data_processed.h5", "r") as f:
    cells = pd.DataFrame({"x_um": f["obs/x_centroid"][:], "y_um": f["obs/y_centroid"][:]},
                         index=[c.decode() for c in f["obs/cell_id"][:]])
groups = pd.read_csv(CELL_GROUPS, index_col="cell_id")["group"]
cells = cells.join(groups.rename("cell_type"), how="inner")
print(f"{len(cells):,} annotated cells")
cells["cell_type"].value_counts()

In [ ]:
RADIUS_UM, N_NICHES = 50, 8

onehot = pd.get_dummies(cells["cell_type"]).to_numpy(dtype=np.float32)
nn = NearestNeighbors(radius=RADIUS_UM).fit(cells[["x_um", "y_um"]])
neigh = nn.radius_neighbors(cells[["x_um", "y_um"]], return_distance=False)
composition = np.stack([onehot[i].mean(0) for i in neigh])
cells["niche"] = "N" + pd.Series(
    KMeans(N_NICHES, n_init=10, random_state=0).fit_predict(composition), index=cells.index).astype(str)

comp = pd.DataFrame(composition, index=cells.index,
                    columns=pd.get_dummies(cells["cell_type"]).columns).groupby(cells["niche"]).mean()
fig, ax = plt.subplots(figsize=(9, 3.5))
im = ax.imshow(comp.to_numpy(), cmap="Blues", aspect="auto")
ax.set_xticks(range(comp.shape[1]), comp.columns, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(comp)), comp.index)
ax.set_title(f"mean cell type composition within {RADIUS_UM} µm, per niche")
plt.colorbar(im, ax=ax, fraction=0.02); plt.tight_layout(); plt.show()

`process_slide_data` reads annotations from an AnnData whose `obs` has a sample column,
`cell_id` and one column per label, so we write the two labels in that form.

In [ ]:
labels = ad.AnnData(obs=pd.DataFrame({
    "sample": pd.Categorical([SECTION] * len(cells)),
    "cell_id": pd.Categorical(cells.index),
    "cell_type": pd.Categorical(cells["cell_type"]),
    "niche": pd.Categorical(cells["niche"]),
}, index=cells.index))
labels_h5ad = OUT / "labels.h5ad"
labels.write_h5ad(labels_h5ad)

LABELS = ("cell_type", "niche")
label_cfg = dict(sample_col="sample", normalize_sample=False,
                 label_specs=tuple((l, l) for l in LABELS))

## 2. Cell crops

For each cell, `process_slide_data` stores the crop origin and a rasterised cell mask at
64 / 128 / 256 px (0.5 µm/px, i.e. 32 / 64 / 128 µm around the cell). It needs no GPU and
is shared by every backbone.

In [ ]:
slide_row = pd.Series({
    "sample_name": SECTION,
    "norm": SECTION,
    "hematoxylin_eosin_image": str(SECTION_DIR / "hematoxylin_eosin.zarr"),
    "spatial_preprocessed": str(SECTION_DIR / "spatial_data_processed.h5"),
    "raw_dir": str(SECTION_DIR),          # folder with cell_boundaries.parquet
})
data_h5 = OUT / "data" / f"{SECTION}.h5"
if not data_h5.exists():
    process_slide_data(slide_row=slide_row, labels_h5ad=str(labels_h5ad),
                       scale_sizes=(64, 128, 256), out_path=data_h5, **label_cfg)

## 3. Embed every cell

`process_slide_features` mask-pools the encoder's patch tokens over each cell. With
`kind="sigmma"` it uses SIGMMA's LoRA-adapted UNI and per-scale projection heads
(`z_sigmma_<scale>`); with `kind="uni"` the raw UNI backbone (`z_uni_<scale>`).

In [ ]:
config = ConfigDict({
    "model": {
        "name": "sigmma",
        "checkpoint_path": str(CKPT),
        "pretrained_image_encoder": "UNI",
        "pretrained_image_encoder_ckpt_path": str(UNI_DIR),
        "image_encoder_input_dim": 1536,
        "image_encoder_output_dim": 512,
        "spatial_encoder_input_dim": 384,
        "spatial_encoder_output_dim": 512,
        "num_resolutions": 3,
        "contrastive_temperature": 0.07,
        "gumbel_temperature": 0.5,
        "edge_threshold": 0.5,
        "candidate_top_k": 8,
        "candidate_sim_threshold": 0.5,
        "dropout": 0.1,
        "activation": "leaky_relu",
        "aggregate": "attn_pool",
        "num_graph_layers": 2,
        "image_mlp_hidden_dim": 2048,
        "image_attn_pool_heads": 8,
        "lora": {"enabled": True, "r": 16, "alpha": 32, "dropout": 0.05,
                 "target_modules": ["qkv", "proj"], "bias": "none", "per_scale": False},
    },
})
sigmma_model = load_model(config, pretrained_image_encoder="UNI",
                          pretrained_ckpt_path=str(CKPT), train=False).to(device).eval()
uni_model = UNI(checkpoint_path=str(UNI_DIR)).to(device).eval()

for kind, backbone in (("sigmma", sigmma_model), ("uni", uni_model)):
    out_path = OUT / kind / f"{SECTION}.h5"
    if not out_path.exists():
        process_slide_features(slide_row=slide_row, data_h5_path=data_h5, backbone_model=backbone,
                               kind=kind, sigmma_model=sigmma_model if kind == "sigmma" else None,
                               encoder_input=224, patch_size=16, batch_size=64, out_path=out_path)

## 4. Train on the left, predict on the right

The section is split at the median x coordinate. Cells within 128 µm of the split (the
largest crop size) are dropped so that no training crop overlaps a test crop.

In [ ]:
SCALES = ("micro", "meso", "macro")

with h5py.File(data_h5) as f:
    d = {"cell_id": f["cell_id"][:], "x": f["x_pixel"][:], "y": f["y_pixel"][:]}
    for l in LABELS:
        d[l] = np.array([s.decode() for s in f[l][:]])
for kind in ("sigmma", "uni"):
    with h5py.File(OUT / kind / f"{SECTION}.h5") as f:
        assert (f["cell_id"][:] == d["cell_id"]).all()
        for s in SCALES:
            d[f"{kind}_{s}"] = f[f"z_{kind}_{s}"][:]
    d[f"{kind}_concat"] = np.concatenate([d[f"{kind}_{s}"] for s in SCALES], axis=1)

GAP_PX = 256   # 128 µm at 0.5 µm/px
x_split = np.median(d["x"])
is_train = d["x"] < x_split - GAP_PX / 2
is_test = d["x"] > x_split + GAP_PX / 2
print(f"train: {is_train.sum():,} cells, test: {is_test.sum():,} cells, "
      f"dropped at the split: {(~is_train & ~is_test).sum():,}")

In [ ]:
FEATURES = ["uni_micro", "uni_macro", "uni_concat",
            "sigmma_micro", "sigmma_meso", "sigmma_macro", "sigmma_concat"]

rows, preds = [], {}
for label in LABELS:
    y_tr, y_te = d[label][is_train], d[label][is_test]
    keep = np.isin(y_te, np.unique(y_tr))     # classes absent from training cannot be predicted
    for feat in FEATURES:
        scaler = StandardScaler().fit(d[feat][is_train])
        clf = LogisticRegression(max_iter=1000, n_jobs=-1).fit(scaler.transform(d[feat][is_train]), y_tr)
        p = clf.predict(scaler.transform(d[feat][is_test]))
        preds[label, feat] = p
        rows.append({"label": label, "features": feat,
                     "balanced_accuracy": balanced_accuracy_score(y_te[keep], p[keep]),
                     "macro_F1": f1_score(y_te[keep], p[keep], average="macro")})
results = pd.DataFrame(rows)
results.pivot(index="features", columns="label", values="macro_F1").loc[FEATURES].round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, label in zip(axes, LABELS):
    r = results[results.label == label].set_index("features").loc[FEATURES]
    ax.barh(FEATURES, r["macro_F1"], color=["#9db4c0" if f.startswith("uni") else "#3d5a80" for f in FEATURES])
    ax.set_title(label); ax.set_xlabel("macro F1 on the held-out region"); ax.set_xlim(0, 1)
axes[0].invert_yaxis(); plt.tight_layout(); plt.show()

## 5. Predictions on the held-out region

In [ ]:
he = zarr.open(slide_row["hematoxylin_eosin_image"], mode="r")
x0 = int(x_split + GAP_PX / 2)
step = 4
crop = np.transpose(np.asarray(he[:, ::step, x0::step]), (1, 2, 0))
extent = [x0, he.shape[2], he.shape[1], 0]
xt, yt = d["x"][is_test], d["y"][is_test]

for label in LABELS:
    classes = sorted(np.unique(d[label]))
    palette = plt.cm.tab20(np.linspace(0, 1, 20))
    cmap = {c: palette[i % 20] for i, c in enumerate(classes)}
    panels = [("ST annotation", d[label][is_test]),
              ("UNI (concat)", preds[label, "uni_concat"]),
              ("SIGMMA (concat)", preds[label, "sigmma_concat"])]
    fig, axes = plt.subplots(3, 1, figsize=(12, 3 * 3 * crop.shape[0] / crop.shape[1] + 1.5))
    for ax, (title, lab) in zip(axes, panels):
        ax.imshow(crop, extent=extent, alpha=0.4)
        ax.scatter(xt, yt, c=[cmap[l] for l in lab], s=0.3, rasterized=True)
        ax.set_title(f"{label}: {title}", fontsize=10); ax.axis("off")
    handles = [plt.Line2D([], [], marker="o", ls="", color=cmap[c], label=c) for c in classes]
    fig.legend(handles=handles, loc="lower center", ncol=min(len(classes), 8), fontsize=8, frameon=False)
    plt.tight_layout(rect=(0, 0.08, 1, 1)); plt.show()